In [1]:
#import the necessary library
import pickle
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


In [2]:
# Load the dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=iris.target)

In [3]:
# Train logistic regression model
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

print(f"Test accuracy: {log_reg.score(X_test, y_test):.2%}")

# Save the model
with open("logistic_model.pkl", "wb") as model_file:
    pickle.dump(log_reg, model_file)

print("Model trained and saved as logistic_model.pkl")

Test accuracy: 96.67%
Model trained and saved as logistic_model.pkl


**Create a Flask Application to Serve the Model**

In [4]:
pip install flask-sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [5]:
from flask import Flask, render_template, request, redirect, url_for
from flask_sqlalchemy import SQLAlchemy
import datetime

In [6]:
#configure the Flask app
app = Flask(__name__)
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:///event_database.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
from flask import Flask, request,jsonify
import math
from pathlib import Path

In [ ]:
# Load your trained logistic regression model
model_path = Path("logistic_model.pkl")

with model_path.open("rb") as model_file:
    model = pickle.load(model_file)

species_names = ["setosa", "versicolor", "virginica"]


@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    if not isinstance(data, dict):
        return jsonify(error="Please provide a JSON object."), 400

    try:
        # Features must follow the same order as the Iris training data
        features = [[
            float(data["sepal_length"]),
            float(data["sepal_width"]),
            float(data["petal_length"]),
            float(data["petal_width"])
        ]]
    except (KeyError, TypeError, ValueError):
        return jsonify(
            error="Provide all four features with numeric values."
        ), 400

    prediction = int(model.predict(features)[0])

    return jsonify(
        prediction=prediction,
        species=species_names[prediction]
    )


if __name__ == "__main__":
    app.run(port=5000, debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [04/Oct/2026 17:39:53] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [04/Oct/2026 17:40:41] "GET / HTTP/1.1" 404 -
127.0.0.1 - - [04/Oct/2026 17:40:41] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [04/Oct/2026 17:42:47] "GET /docs HTTP/1.1" 404 -
127.0.0.1 - - [04/Oct/2026 17:43:25] "GET /predict HTTP/1.1" 405 -
127.0.0.1 - - [04/Oct/2026 17:43:56] "GET /post HTTP/1.1" 404 -
127.0.0.1 - - [04/Oct/2026 17:44:40] "GET /predict HTTP/1.1" 405 -
